# Superhedging experiment (local version)

Run the cells top to bottom. `fast_solvers.py` is embedded in the notebook and written out automatically. Results are written to `awdro_results/time_grid_<id>/` next to this notebook.
Finished seeds are checkpointed, so if the run is interrupted you can re-run from the setup cell and completed seeds are skipped.

Set `NEIGHBORS` in the config cell: `None` is the exact problem; an integer L restricts the adversary to the L nearest states per date (sparse, much faster, smaller ambiguity set so lower values).

This is the fast configuration: sparse couplings (`NEIGHBORS = 3`), 2-6 trading intervals, rectangular solved up to `RECT_MAX_STEPS = 4`, nonrectangular for all horizons.

In [ ]:
# Run once, then restart the kernel. Uncomment to install.
# %pip install numpy pandas scipy torch cvxpy highspy matplotlib ipython

In [ ]:
from pathlib import Path
import subprocess
import sys

PROJECT_DIR = Path.cwd()
RESULTS_ROOT = PROJECT_DIR / "awdro_results"   # all results are saved here

# If you already have the repo locally, set this to the folder that contains
# hedging/, "path simulation"/, etc. Otherwise leave None and it is cloned.
LOCAL_REPO = None  # e.g. Path(r"C:\Users\me\projects\awdro")

REPOSITORY_URL = "https://github.com/jcvetinovska/awdro.git"
CHECKOUT = Path(LOCAL_REPO) if LOCAL_REPO else PROJECT_DIR / "awdro_repo"

if not CHECKOUT.exists():
    subprocess.run(["git", "clone", REPOSITORY_URL, str(CHECKOUT)], check=True)

# To update the code later:
# subprocess.run(["git", "-C", str(CHECKOUT), "pull", "--ff-only"], check=True)
# then restart the kernel.

required_folders = [
    "hedging",
    "path simulation",
    "inner maximization",
    "outer minimization",
]

# Supports either repository root/hedging or repository root/awdro/hedging.
root = next(
    (
        candidate
        for candidate in [CHECKOUT, CHECKOUT / "awdro"]
        if all((candidate / name).is_dir() for name in required_folders)
    ),
    None,
)

if root is None:
    raise FileNotFoundError(
        "Cannot locate the project folders. "
        "Set LOCAL_REPO to their parent directory."
    )

for folder in [PROJECT_DIR, root] + [root / name for name in required_folders]:
    if str(folder) not in sys.path:
        sys.path.insert(0, str(folder))

try:
    CODE_VERSION = subprocess.check_output(
        ["git", "-C", str(CHECKOUT), "rev-parse", "HEAD"],
        text=True,
    ).strip()
except (subprocess.CalledProcessError, FileNotFoundError):
    CODE_VERSION = "unknown"   # not a git checkout

print("Project:", root)
print("Commit:", CODE_VERSION)


In [ ]:
# Writes fast_solvers.py next to the notebook, so no separate file is needed.
from pathlib import Path
FAST_SOLVERS_SOURCE = r'''"""Fast, optionally sparse, drop-in solvers for the rectangular / nonrectangular study.

Nothing in the repository files is modified.  Two classes are provided:

    FastRectangularSolver(model, times, budget, discretization, neighbors=None)
    FastRelaxedSuperhedgingSolver(problem=..., inner_solver=..., penalty=...)

Exact mode (neighbors=None) solves exactly the same problem as the original
code: the adversary may move to any combination of reference-tree states.  The
speedups there are vectorised setup, sparse matrices handed straight to HiGHS
instead of cvxpy expressions, and no per-history reachability LPs.

Sparse mode (neighbors=L) restricts the coupling: reference path i may only be
coupled to adversary paths whose state at every date is one of the L candidate
states nearest to i's own state at that date (its own state included).  The
adversary path space is then a union of small local products instead of the
full product, so it is exponentially smaller.  This shrinks the ambiguity set,
so values change; neighbors=None is the exact problem.
"""

import numpy as np
import torch
from scipy.optimize import linprog
from scipy.sparse import csr_matrix, coo_matrix, hstack, vstack, identity

from RectangularSolver import RectangularSolver
from RelaxedSuperhedgingSolver import RelaxedSuperhedgingSolver


# --------------------------------------------------------------------------
# Transport model: reference paths, adversary paths and the (sparse) coupling
# --------------------------------------------------------------------------
class FastRectangularSolver(RectangularSolver):
    """Holds the finite transport model.  Couplings are indexed by *pairs*
    (reference path i, adversary path j); in exact mode every pair exists."""

    def __init__(self, reference_model, trading_times, budget, discretization,
                 neighbors=None):
        self.reference_model = reference_model
        self.trading_times = trading_times
        self.budget = budget
        self.discretization = discretization
        self.neighbors = neighbors

        self.state_space = reference_model.is_valid_state
        self._validate_inputs()
        if neighbors is not None and (
            not isinstance(neighbors, (int, np.integer)) or neighbors < 1
        ):
            raise ValueError("neighbors must be None or a positive integer.")

        self.reference_nodes = self._build_reference_model()
        self.adversarial_candidates = self._build_adversarial_candidates()
        self.reference_paths, self.reference_probabilities = (
            self._extract_reference_paths()
        )

        self._build_pairs()
        self.transport_costs = self._pair_costs()

        reference_A, reference_b = self._reference_rows()
        causality_A, causality_b = self._causality_rows()
        self.A_eq = vstack([reference_A, causality_A], format="csr")
        self.b_eq = np.concatenate([reference_b, causality_b])

        self.A_ub = csr_matrix(self.transport_costs.reshape(1, -1))
        self.b_ub = np.array([self.budget ** 2], dtype=float)

    # ---- adversary paths and pairs -----------------------------------------
    def _build_pairs(self):
        n_times = len(self.trading_times)
        T = n_times - 1
        refs = self.reference_paths
        n = refs.shape[0]
        cand = self.adversarial_candidates

        lookup, tables = {}, {}
        for t in range(1, n_times):
            C = cand[t]
            lookup[t] = {tuple(row): k for k, row in enumerate(C)}
            if self.neighbors is None:
                tables[t] = None
            else:
                D = ((C[:, None, :] - C[None, :, :]) ** 2).sum(-1)
                order = np.argsort(D, axis=1, kind="stable")
                tables[t] = order[:, : min(int(self.neighbors), len(C))]

        ref_index = np.empty((n, T), dtype=np.int64)
        for t in range(1, n_times):
            for i in range(n):
                ref_index[i, t - 1] = lookup[t][tuple(refs[i, t])]

        blocks, ref_of = [], []
        for i in range(n):
            axes = [
                np.arange(len(cand[t])) if tables[t] is None
                else tables[t][ref_index[i, t - 1]]
                for t in range(1, n_times)
            ]
            grid = np.stack(
                np.meshgrid(*axes, indexing="ij"), axis=-1
            ).reshape(-1, T)
            blocks.append(grid)
            ref_of.append(np.full(len(grid), i, dtype=np.int64))

        stacked = np.concatenate(blocks)
        unique_paths, inverse = np.unique(
            stacked, axis=0, return_inverse=True
        )
        inverse = np.asarray(inverse).ravel()

        m = len(unique_paths)
        adv = np.empty((m, n_times, 2), dtype=float)
        adv[:, 0, :] = self.reference_nodes[0]["state"]
        for t in range(1, n_times):
            adv[:, t, :] = cand[t][unique_paths[:, t - 1]]

        self.adversarial_paths = adv
        self.pair_ref = np.concatenate(ref_of)
        self.pair_adv = inverse
        self.n_pairs = len(inverse)
        # pairs of each reference path are contiguous
        counts = np.bincount(self.pair_ref, minlength=n)
        self._ref_ptr = np.concatenate([[0], np.cumsum(counts)])

    def _pair_costs(self):
        refs = self.reference_paths[:, 1:, :]
        adv = self.adversarial_paths[:, 1:, :]
        out = np.empty(self.n_pairs)
        step = 200_000
        for a in range(0, self.n_pairs, step):
            b = min(a + step, self.n_pairs)
            d = refs[self.pair_ref[a:b]] - adv[self.pair_adv[a:b]]
            out[a:b] = np.einsum("ptk,ptk->p", d, d)
        return out

    # ---- constraints --------------------------------------------------------
    def _reference_rows(self):
        n = self.reference_paths.shape[0]
        A = coo_matrix(
            (np.ones(self.n_pairs), (self.pair_ref, np.arange(self.n_pairs))),
            shape=(n, self.n_pairs),
        ).tocsr()
        return A, self.reference_probabilities.copy()

    def _causality_rows(self):
        n = self.reference_paths.shape[0]
        m = self.adversarial_paths.shape[0]
        P = self.n_pairs
        probs = self.reference_probabilities
        if not np.isfinite(probs).all() or np.any(probs <= 0):
            raise ValueError(
                "Reference-path probabilities must be finite and positive."
            )

        by_ref = [
            np.arange(self._ref_ptr[i], self._ref_ptr[i + 1]) for i in range(n)
        ]
        rows, cols, vals = [], [], []
        next_row = 0

        for t in range(1, len(self.trading_times) - 1):
            _, rg = np.unique(
                self.reference_paths[:, : t + 1, :].reshape(n, -1),
                axis=0, return_inverse=True,
            )
            _, ag = np.unique(
                self.adversarial_paths[:, : t + 1, :].reshape(m, -1),
                axis=0, return_inverse=True,
            )
            rg = np.asarray(rg).ravel()
            pair_group = np.asarray(ag).ravel()[self.pair_adv]

            for g in range(int(rg.max()) + 1):
                members = np.flatnonzero(rg == g)
                if members.size < 2:
                    continue
                a = int(members[0])
                ga = pair_group[by_ref[a]]
                for i in members[1:]:
                    gi = pair_group[by_ref[i]]
                    groups = np.union1d(gi, ga)
                    rows.append(next_row + np.searchsorted(groups, gi))
                    cols.append(by_ref[i])
                    vals.append(np.full(gi.size, 1.0 / probs[i]))
                    rows.append(next_row + np.searchsorted(groups, ga))
                    cols.append(by_ref[a])
                    vals.append(np.full(ga.size, -1.0 / probs[a]))
                    next_row += groups.size

        if rows:
            A = coo_matrix(
                (np.concatenate(vals),
                 (np.concatenate(rows), np.concatenate(cols))),
                shape=(next_row, P),
            ).tocsr()
        else:
            A = csr_matrix((0, P), dtype=float)
        return A, np.zeros(next_row)


# --------------------------------------------------------------------------
# Outer problems: sparse matrices straight into HiGHS
# --------------------------------------------------------------------------
class FastRelaxedSuperhedgingSolver(RelaxedSuperhedgingSolver):

    # Refuse LPs with more constraint rows than this, so an infeasible grid
    # fails at once with a clear message instead of exhausting memory.
    max_lp_rows = 5_000_000
    # "highs" lets HiGHS choose; "highs-ipm" / "highs-ds" force a method.
    lp_method = "highs"

    def _check_size(self, rows, label):
        if rows > self.max_lp_rows:
            raise RuntimeError(
                f"{label} LP would have {rows:,} rows "
                f"(limit {self.max_lp_rows:,}); skipped."
            )

    def _solve_lp(self, c, A, b, lower, upper, label):
        result = linprog(
            c, A_ub=A, b_ub=b,
            bounds=np.column_stack([lower, upper]),
            method=self.lp_method,
        )
        if result.status != 0 or result.x is None:
            raise RuntimeError(f"{label} LP failed: {result.message}")
        return result

    def _prepare(self, tm, capital_bounds, holding_bounds):
        bounds = np.asarray([capital_bounds, holding_bounds], dtype=float)
        if bounds.shape != (2, 2) or not np.isfinite(bounds).all():
            raise ValueError("Provide two finite bounds for capital and holding.")
        if np.any(bounds[:, 0] >= bounds[:, 1]):
            raise ValueError("Each lower bound must be below its upper bound.")

        paths = tm.adversarial_paths
        with torch.no_grad():
            payoff_tensor = self.problem.payoffs(
                torch.tensor(paths, dtype=torch.float64)
            )
        payoffs = payoff_tensor.detach().cpu().numpy()
        if payoffs.shape != (paths.shape[0],) or not np.isfinite(payoffs).all():
            raise ValueError("Provide one finite payoff per adversarial path.")

        G, histories_by_time, offsets = self._build_adaptive_gains_matrix(paths)
        return bounds, paths, payoffs, G, histories_by_time, offsets

    @staticmethod
    def _result(x, nh, histories_by_time, offsets, value, shortfall):
        return {
            "capital": float(x[0]),
            "holdings": np.asarray(x[1:1 + nh], dtype=float).copy(),
            "histories_by_time": histories_by_time,
            "offsets": offsets,
            "value": float(value),
            "worst_case_shortfall": float(shortfall),
        }

    # ---- nonrectangular -----------------------------------------------------
    def solve_adaptive_hedge_nonrectangular(
        self, transport_model, capital_bounds, holding_bounds
    ):
        tm = transport_model
        bounds, paths, payoffs, G, histories_by_time, offsets = (
            self._prepare(tm, capital_bounds, holding_bounds)
        )
        m = paths.shape[0]
        nh = G.shape[1]
        P = tm.n_pairs
        A_eq, b_eq, A_ub, b_ub = tm.A_eq, tm.b_eq, tm.A_ub, tm.b_ub
        R, Rb = A_eq.shape[0], A_ub.shape[0]
        self._check_size(P + m, "Nonrectangular")

        i_s = 1 + nh
        i_mu = i_s + m
        i_beta = i_mu + R
        n_vars = i_beta + Rb

        c = np.zeros(n_vars)
        c[0] = 1.0
        c[i_mu:i_mu + R] = self.penalty * b_eq
        c[i_beta:] = self.penalty * b_ub

        gain_rows = hstack(
            [
                csr_matrix(-np.ones((m, 1))),
                -G,
                -identity(m, format="csr"),
                csr_matrix((m, R)),
                csr_matrix((m, Rb)),
            ],
            format="csr",
        )
        select = csr_matrix(
            (np.ones(P), (np.arange(P), tm.pair_adv)), shape=(P, m)
        )
        dual_rows = hstack(
            [csr_matrix((P, 1 + nh)), select, -A_eq.T, -A_ub.T],
            format="csr",
        )

        A = vstack([gain_rows, dual_rows], format="csr")
        b = np.concatenate([-payoffs, np.zeros(P)])

        lower = np.concatenate([
            [bounds[0, 0]], np.full(nh, bounds[1, 0]),
            np.zeros(m), np.full(R, -np.inf), np.zeros(Rb),
        ])
        upper = np.concatenate([
            [bounds[0, 1]], np.full(nh, bounds[1, 1]),
            np.full(m, np.inf), np.full(R, np.inf), np.full(Rb, np.inf),
        ])

        res = self._solve_lp(c, A, b, lower, upper, "Nonrectangular")
        x = res.x
        shortfall = b_eq @ x[i_mu:i_mu + R] + b_ub @ x[i_beta:]
        return self._result(x, nh, histories_by_time, offsets, res.fun, shortfall)

    # ---- rectangular ---------------------------------------------------------
    def solve_adaptive_hedge_rectangular(
        self, transport_model, capital_bounds, holding_bounds
    ):
        tm = transport_model
        bounds, paths, payoffs, G, histories_by_time, offsets = (
            self._prepare(tm, capital_bounds, holding_bounds)
        )
        m, n_times, _ = paths.shape
        nh = G.shape[1]
        P = tm.n_pairs
        i_s = 1 + nh

        gain_rows = hstack(
            [csr_matrix(-np.ones((m, 1))), -G, -identity(m, format="csr")],
            format="csr",
        )

        if tm.budget == 0:
            return self._rectangular_zero_budget(
                tm, bounds, paths, payoffs, gain_rows, nh, m,
                histories_by_time, offsets,
            )

        A_eq, b_eq, A_ub, b_ub = tm.A_eq, tm.b_eq, tm.A_ub, tm.b_ub
        if A_ub.shape[0] != 1:
            raise NotImplementedError("Expected a single global budget row.")
        R = A_eq.shape[0]

        group_id, n_groups = [], []
        for t in range(n_times - 1):
            _, g = np.unique(
                paths[:, :t + 1, :].reshape(m, -1), axis=0, return_inverse=True
            )
            g = np.asarray(g).ravel()
            group_id.append(g)
            n_groups.append(int(g.max()) + 1)

        n_prefix = int(sum(n_groups))
        self._check_size(n_prefix * (P + 1), "Rectangular")

        # Every history in the candidate space is reachable when budget > 0,
        # so the per-history reachability LPs of the original code are skipped.
        block = R + 3                         # e (R + 1), budget dual, value
        group_start, prefix_index = [], {}
        position, count = i_s + m, 0
        for t in range(n_times - 1):
            starts = []
            for g in range(n_groups[t]):
                starts.append(position)
                prefix_index[(t, g)] = count
                position += block
                count += 1
            group_start.append(np.asarray(starts))
        n_vars = position

        # value variable of the child history reached by each pair
        child_cols = []
        for t in range(n_times - 1):
            if t + 1 == n_times - 1:
                child_cols.append(i_s + tm.pair_adv)
            else:
                child_cols.append(
                    group_start[t + 1][group_id[t + 1][tm.pair_adv]] + R + 2
                )

        Bt = vstack(
            [-A_eq.T, csr_matrix(b_eq.reshape(1, -1))], format="coo"
        )
        ub_column = np.concatenate([-A_ub.toarray().ravel(), [b_ub[0]]])
        ub_rows = np.flatnonzero(ub_column)
        ub_vals = ub_column[ub_rows]

        rows, cols, vals = [], [], []
        node_rows, node_cols, node_vals = [], [], []

        for t in range(n_times - 1):
            pair_group = group_id[t][tm.pair_adv]
            order = np.argsort(pair_group, kind="stable")
            bounds_g = np.searchsorted(
                pair_group[order], np.arange(n_groups[t] + 1)
            )
            for g in range(n_groups[t]):
                p = prefix_index[(t, g)]
                start = group_start[t][g]
                row0 = p * (P + 1)
                hist = order[bounds_g[g]:bounds_g[g + 1]]

                rows.append(Bt.row + row0)
                cols.append(Bt.col + start)
                vals.append(Bt.data)

                rows.append(row0 + hist)                      # normalisation
                cols.append(np.full(hist.size, start + R))
                vals.append(np.full(hist.size, -1.0))

                rows.append(ub_rows + row0)                   # budget dual
                cols.append(np.full(ub_rows.size, start + R + 1))
                vals.append(ub_vals)

                rows.append(row0 + hist)                      # child values
                cols.append(child_cols[t][hist])
                vals.append(np.ones(hist.size))

                node_rows += [p, p]                           # e[R] - v <= 0
                node_cols += [start + R, start + R + 2]
                node_vals += [1.0, -1.0]

        main_rows = n_prefix * (P + 1)
        dual = coo_matrix(
            (np.concatenate(vals), (np.concatenate(rows), np.concatenate(cols))),
            shape=(main_rows, n_vars),
        ).tocsr()
        node = coo_matrix(
            (node_vals, (node_rows, node_cols)), shape=(n_prefix, n_vars)
        ).tocsr()
        gain = hstack(
            [gain_rows, csr_matrix((m, n_vars - gain_rows.shape[1]))],
            format="csr",
        )

        A = vstack([gain, dual, node], format="csr")
        b = np.concatenate([-payoffs, np.zeros(main_rows + n_prefix)])

        lower = np.full(n_vars, -np.inf)
        upper = np.full(n_vars, np.inf)
        lower[0], upper[0] = bounds[0]
        lower[1:1 + nh], upper[1:1 + nh] = bounds[1]
        lower[i_s:i_s + m] = 0.0
        for t in range(n_times - 1):
            lower[group_start[t] + R + 1] = 0.0

        root_value = group_start[0][0] + R + 2
        c = np.zeros(n_vars)
        c[0] = 1.0
        c[root_value] = self.penalty

        res = self._solve_lp(c, A, b, lower, upper, "Rectangular")
        return self._result(
            res.x, nh, histories_by_time, offsets, res.fun, res.x[root_value]
        )

    def _rectangular_zero_budget(
        self, tm, bounds, paths, payoffs, gain_rows, nh, m,
        histories_by_time, offsets,
    ):
        lookup = {np.ascontiguousarray(p).tobytes(): j for j, p in enumerate(paths)}
        weights = np.zeros(m)
        for path, probability in zip(
            tm.reference_paths, tm.reference_probabilities
        ):
            weights[lookup[np.ascontiguousarray(path).tobytes()]] += probability

        c = np.concatenate([[1.0], np.zeros(nh), self.penalty * weights])
        lower = np.concatenate(
            [[bounds[0, 0]], np.full(nh, bounds[1, 0]), np.zeros(m)]
        )
        upper = np.concatenate(
            [[bounds[0, 1]], np.full(nh, bounds[1, 1]), np.full(m, np.inf)]
        )
        res = self._solve_lp(
            c, gain_rows, -payoffs, lower, upper, "Rectangular (zero budget)"
        )
        shortfall = weights @ res.x[1 + nh:]
        return self._result(
            res.x, nh, histories_by_time, offsets, res.fun, shortfall
        )
'''
Path(PROJECT_DIR, 'fast_solvers.py').write_text(FAST_SOLVERS_SOURCE)
import importlib; importlib.invalidate_caches()
print('Wrote', Path(PROJECT_DIR, 'fast_solvers.py'))


In [ ]:
from pathlib import Path
import hashlib
import json
import platform
import shutil
import time
from importlib.metadata import version

import numpy as np
import pandas as pd
import torch
import cvxpy as cp
import matplotlib.pyplot as plt
from IPython.display import display

from Heston import Heston
from SuperhedgingProblem import SuperhedgingProblem
from NonrectangularSolver import NonrectangularSolver
from RectangularSolver import RectangularSolver
from RelaxedSuperhedgingSolver import RelaxedSuperhedgingSolver
from fast_solvers import FastRectangularSolver, FastRelaxedSuperhedgingSolver

if "HIGHS" not in cp.installed_solvers():
    raise RuntimeError("Restart the kernel after installing highspy.")

# Pilot run. Increase to 50 or 100 after measuring runtime.
N_SEEDS = 30
SEEDS = list(range(1000, 1000 + N_SEEDS))

# FAST SETTINGS
# NEIGHBORS = L: each reference path may only be coupled to adversary paths whose
# state at every date is one of the L nearest candidate states (sparse coupling).
# This is a SMALLER ambiguity set than the exact problem (NEIGHBORS = None), so
# values are lower, rectangular especially. Compare results only at equal L.
# Measured (seed 1000, 4 steps; exact = 1.601 / 8.977): L=3 -> 1.49 / 5.87,
# L=4 -> 1.50 / 8.25, L=6 -> 1.50 / 8.68.
NEIGHBORS = 3

# The rectangular LP is the bottleneck (it did not finish at 5 steps even with
# L=3 in a 4-minute test), so it is only run up to RECT_MAX_STEPS. Above that
# only the nonrectangular value is computed (6 steps, L=3: ~20 s per seed) and
# the gap is left empty.
RECT_MAX_STEPS = 4

# Wall-clock cap: no new run is started after this many minutes. Seeds are looped
# outermost, so every horizon ends up with the same number of seeds.
TIME_BUDGET_MINUTES = 30
N_STEPS_LIST = [2, 3, 4, 5, 6]
BRANCHES = 2

MATURITY = 1.0
BUDGET = 1.0
PENALTY = 10.0
STRIKE = 100.0

CAPITAL_BOUNDS = (0.0, 50.0)
HOLDING_BOUNDS = (-2.0, 2.0)

ABS_TOL = 1e-5
REL_TOL = 1e-5

MODEL_PARAMETERS = {
    "s0": 100.0,
    "v0": 0.04,
    "mu": 0.0,
    "kappa": 2.0,
    "theta": 0.04,
    "vol_of_vol": 0.3,
    "rho": -0.7,
}

# Increment if you change the notebook's experimental logic.
EXPERIMENT_VERSION = "1"

config = {
    "experiment_version": EXPERIMENT_VERSION,
    "git_commit": CODE_VERSION,
    "model": MODEL_PARAMETERS,
    "maturity": MATURITY,
    "budget": BUDGET,
    "penalty": PENALTY,
    "strike": STRIKE,
    "branches": BRANCHES,
    "neighbors": NEIGHBORS,
    "rect_max_steps": RECT_MAX_STEPS,
    "capital_bounds": CAPITAL_BOUNDS,
    "holding_bounds": HOLDING_BOUNDS,
    "payoff": "European call",
    "cost": "unweighted sum of squared stock and variance differences",
    "abs_tol": ABS_TOL,
    "rel_tol": REL_TOL,
    "python": platform.python_version(),
    "packages": {
        package: version(package)
        for package in [
            "numpy", "pandas", "scipy", "torch",
            "cvxpy", "highspy", "matplotlib",
        ]
    },
}

# Seeds and time grids are excluded so completed runs can be reused.
study_id = hashlib.sha256(
    json.dumps(config, sort_keys=True).encode()
).hexdigest()[:12]

results_root = RESULTS_ROOT
output_dir = results_root / f"time_grid_{study_id}"
output_dir.mkdir(parents=True, exist_ok=True)


def save_json(path, value):
    temporary = path.with_suffix(path.suffix + ".tmp")
    with open(temporary, "w") as file:
        json.dump(value, file, indent=2)
    temporary.replace(path)


save_json(output_dir / "configuration.json", config)
save_json(
    output_dir / "requested_runs.json",
    {"seeds": SEEDS, "n_steps": N_STEPS_LIST},
)

print("Results folder:", output_dir.resolve())
print("Paired experiments:", len(SEEDS) * len(N_STEPS_LIST))

In [ ]:
def to_array(x):
    """Convert to float array; fall back to object array if ragged."""
    try:
        return np.asarray(x, dtype=float)
    except ValueError:
        return np.asarray(x, dtype=object)   # load with allow_pickle=True


def call_payoff(paths):
    return torch.clamp(paths[:, -1, 0] - STRIKE, min=0.0)


def run_pair(seed, n_steps):
    started = time.perf_counter()

    row = {
        "seed": int(seed),
        "n_steps": int(n_steps),
        "status": "failed",
        "stage": "construction",
        "error": "",
    }

    try:
        model = Heston(**MODEL_PARAMETERS)
        times = np.linspace(0.0, MATURITY, n_steps + 1)

        problem = SuperhedgingProblem(
            payoff=call_payoff,
            trading_times=times,
        )

        discretization = {
            "seed": int(seed),
            "reference_branches": BRANCHES,
        }

        rectangular = FastRectangularSolver(
            model, times, BUDGET, discretization, neighbors=NEIGHBORS
        )
        nonrectangular = NonrectangularSolver(
            model, times, BUDGET, discretization
        )

        row["neighbors"] = -1 if NEIGHBORS is None else int(NEIGHBORS)
        row["pairs"] = int(rectangular.n_pairs)
        row["stage"] = "shared_model_check"

        np.testing.assert_array_equal(
            np.asarray([
                node["state"] for node in rectangular.reference_nodes
            ]),
            np.asarray([
                node["state"] for node in nonrectangular.reference_nodes
            ]),
        )

        for t in range(n_steps + 1):
            np.testing.assert_array_equal(
                rectangular.adversarial_candidates[t],
                nonrectangular.adversarial_candidates[t],
            )

        outer = FastRelaxedSuperhedgingSolver(
            problem=problem,
            inner_solver=nonrectangular,
            penalty=PENALTY,
        )

        methods = {
            "nonrectangular": outer.solve_adaptive_hedge_nonrectangular,
        }
        run_rect = n_steps <= RECT_MAX_STEPS
        if run_rect:
            methods["rectangular"] = outer.solve_adaptive_hedge_rectangular

        for name, method in methods.items():
            row["stage"] = f"{name}_optimization"
            solve_started = time.perf_counter()

            result = method(
                rectangular, CAPITAL_BOUNDS, HOLDING_BOUNDS
            )

            row[f"{name}_seconds"] = (
                time.perf_counter() - solve_started
            )

            value = float(result["value"])
            capital = float(result["capital"])
            shortfall = float(result["worst_case_shortfall"])
            holdings = np.asarray(result["holdings"], dtype=float)

            if not np.isfinite([value, capital, shortfall]).all():
                raise RuntimeError(f"{name}: nonfinite scalar result.")

            if holdings.size == 0 or not np.isfinite(holdings).all():
                raise RuntimeError(f"{name}: invalid holdings.")

            np.testing.assert_allclose(
                value,
                capital + PENALTY * shortfall,
                atol=ABS_TOL,
                rtol=REL_TOL,
            )

            capital_tol = ABS_TOL + REL_TOL * max(
                1.0, *map(abs, CAPITAL_BOUNDS)
            )
            holding_tol = ABS_TOL + REL_TOL * max(
                1.0, *map(abs, HOLDING_BOUNDS)
            )
            value_tol = ABS_TOL + REL_TOL * max(1.0, abs(value))

            if shortfall < -value_tol:
                raise RuntimeError(f"{name}: negative shortfall.")

            if not (
                CAPITAL_BOUNDS[0] - capital_tol
                <= capital
                <= CAPITAL_BOUNDS[1] + capital_tol
            ):
                raise RuntimeError(f"{name}: capital bounds violated.")

            if (
                (holdings < HOLDING_BOUNDS[0] - holding_tol).any()
                or (holdings > HOLDING_BOUNDS[1] + holding_tol).any()
            ):
                raise RuntimeError(f"{name}: holding bounds violated.")

            row[f"{name}_value"] = value
            row[f"{name}_capital"] = capital
            row[f"{name}_shortfall"] = shortfall

            row[f"{name}_capital_at_bound"] = bool(
                np.isclose(
                    capital, CAPITAL_BOUNDS,
                    atol=capital_tol, rtol=0.0,
                ).any()
            )

            row[f"{name}_holding_bound_fraction"] = float(
                (
                    np.isclose(
                        holdings, HOLDING_BOUNDS[0],
                        atol=holding_tol, rtol=0.0,
                    )
                    | np.isclose(
                        holdings, HOLDING_BOUNDS[1],
                        atol=holding_tol, rtol=0.0,
                    )
                ).mean()
            )

            arrays = {
                "capital": np.asarray(capital),
                "holdings": holdings,
                "offsets": to_array(result["offsets"]),
                "trading_times": times,
            }
            arrays.update({
                f"histories_t{t}": to_array(histories)
                for t, histories in enumerate(result["histories_by_time"])
            })

            np.savez_compressed(
                output_dir / f"T{n_steps}_seed{seed}_{name}_hedge.npz",
                **arrays,
            )

        row["stage"] = "gap_check"

        if not run_rect:
            row["gap"] = float("nan")
            row["positive_above_tolerance"] = False
        else:
            gap = row["rectangular_value"] - row["nonrectangular_value"]
            tolerance = ABS_TOL + REL_TOL * max(
                1.0,
                abs(row["rectangular_value"]),
                abs(row["nonrectangular_value"]),
            )

            row["gap"] = gap
            row["gap_tolerance"] = tolerance
            row["positive_above_tolerance"] = bool(gap > tolerance)

            if gap < -tolerance:
                raise RuntimeError(
                    f"Gap {gap:.6g} is below -tolerance {-tolerance:.6g}."
                )

            if BUDGET == 0 and abs(gap) > tolerance:
                raise RuntimeError("Zero-budget objectives disagree.")


        row["status"] = "ok"
        row["stage"] = "complete"

    except Exception as exc:
        row["error"] = f"{type(exc).__name__}: {exc}"

    row["total_seconds"] = time.perf_counter() - started
    return row

In [ ]:
records = []
run_started = time.perf_counter()
stopped_early = False

for seed in SEEDS:
    for n_steps in N_STEPS_LIST:
        if time.perf_counter() - run_started > 60 * TIME_BUDGET_MINUTES:
            stopped_early = True
            break
        checkpoint = output_dir / f"T{n_steps}_seed{seed}.json"
        row = None

        if checkpoint.exists():
            with open(checkpoint) as file:
                cached = json.load(file)

            if cached["status"] == "ok":
                row = cached

        if row is None:
            print(
                f"Running {n_steps} intervals, seed {seed}...",
                flush=True,
            )
            row = run_pair(seed, n_steps)
            save_json(checkpoint, row)

        records.append(row)

        # Save the combined table after every experiment.
        pd.DataFrame(records).to_csv(
            output_dir / "results.csv",
            index=False,
        )

        print(
            f"  {row['status']} | "
            f"{row['total_seconds']:.1f} seconds"
        )

        if row["status"] != "ok":
            print(f"  {row['stage']}: {row['error']}")

results = pd.DataFrame(records)

display(
    results.groupby(["n_steps", "status"])
    .size()
    .unstack(fill_value=0)
)
if stopped_early:
    print(f"Time budget of {TIME_BUDGET_MINUTES} min reached; stopped early. "
          f"Seeds completed: {results.groupby('n_steps').size().to_dict()}")
print(f"Total run time: {(time.perf_counter() - run_started)/60:.1f} min")


In [ ]:
if results.empty or "status" not in results:
    raise RuntimeError(
        "No results were produced. Check the errors printed above."
    )

summary_rows = []

for n_steps, group in results.groupby("n_steps"):
    good_group = group.loc[group["status"] == "ok"]

    item = {
        "n_steps": int(n_steps),
        "attempted": len(group),
        "successful": len(good_group),
        "failed": len(group) - len(good_group),
    }

    if len(good_group) >= 1:
        item["mean_nonrectangular_value"] = good_group["nonrectangular_value"].mean()
        item["median_seconds"] = good_group["total_seconds"].median()
    if len(good_group["gap"].dropna()) >= 2:
        gaps = good_group["gap"].dropna().to_numpy(dtype=float)
        rng = np.random.default_rng(2026 + int(n_steps))

        indices = rng.integers(
            0, len(gaps), size=(10_000, len(gaps))
        )
        bootstrap_means = gaps[indices].mean(axis=1)
        low, high = np.quantile(bootstrap_means, [0.025, 0.975])

        item.update({
            "mean_gap": gaps.mean(),
            "median_gap": np.median(gaps),
            "ci_low": low,
            "ci_high": high,
            "positive_fraction": (
                good_group["positive_above_tolerance"].mean()
            ),
            "mean_nonrectangular_value": (
                good_group["nonrectangular_value"].mean()
            ),
            "mean_rectangular_value": (
                good_group["rectangular_value"].mean()
            ),
            "median_seconds": good_group["total_seconds"].median(),
        })

    summary_rows.append(item)

summary = pd.DataFrame(summary_rows)
summary.to_csv(output_dir / "summary.csv", index=False)
display(summary)

if (summary["failed"] > 0).any():
    print(
        "Warning: summaries and plots use successful runs only. "
        "Investigate and report failures."
    )

if N_SEEDS < 30:
    print("This is a runtime pilot, not the final presentation sample.")

In [ ]:
plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 12,
    "axes.titlesize": 17,
    "axes.labelsize": 12,
    "text.color": "#192332",
    "axes.labelcolor": "#394455",
    "xtick.color": "#657083",
    "ytick.color": "#657083",
    "axes.edgecolor": "#CBD1D9",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.dpi": 300,
})

COLORS = ["#2563EB", "#0D9488", "#9333EA", "#EA580C"]

good = results.loc[results["status"] == "ok"].copy()

if "mean_gap" not in summary:
    raise RuntimeError("At least two successful runs per grid are needed.")

plot_summary = summary.dropna(subset=["mean_gap"]).sort_values("n_steps")

subtitle = (
    f"European call · adaptive hedge · ε = {BUDGET:g} · "
    f"κ = {PENALTY:g} · maturity = {MATURITY:g}"
)

failure_note = (
    " Successful runs only; failures recorded."
    if (summary["failed"] > 0).any() else ""
)


def save_figure(fig, filename):
    fig.text(
        0.01, 0.01,
        "Finite-model study · unweighted squared path cost." + failure_note,
        fontsize=9,
        color="#778193",
    )
    fig.tight_layout(rect=[0, 0.06, 1, 0.92])
    fig.savefig(output_dir / f"{filename}.pdf", bbox_inches="tight")
    fig.savefig(output_dir / f"{filename}.png", bbox_inches="tight")
    plt.show()

In [ ]:
# Figure 1: raw gaps, means and uncertainty intervals.

fig, ax = plt.subplots(figsize=(9, 5.2))

for index, (_, stats) in enumerate(plot_summary.iterrows()):
    steps = int(stats["n_steps"])
    color = COLORS[index % len(COLORS)]

    gaps = good.loc[good["n_steps"] == steps, "gap"].to_numpy()
    jitter = np.random.default_rng(steps).uniform(-0.12, 0.12, len(gaps))

    ax.scatter(
        steps + jitter, gaps,
        s=24, color=color, alpha=0.25, edgecolors="none",
    )
    ax.vlines(
        steps, stats["ci_low"], stats["ci_high"],
        color=color, linewidth=3,
    )
    ax.scatter(
        steps, stats["mean_gap"],
        color=color, s=85, edgecolors="white", zorder=4,
    )

ax.axhline(0, color="#B0B8C3", linewidth=0.8)
ax.set_xticks(plot_summary["n_steps"])
ax.set_xlabel("Trading intervals")
ax.set_ylabel("Rectangular − nonrectangular objective")
ax.set_title("The cost of rectangularization", loc="left", pad=18)
ax.grid(axis="y", color="#ECEFF3")
ax.set_axisbelow(True)

ax.text(
    0.02, 0.97,
    "Faint points: seeds\nSolid points: means\nLines: 95% bootstrap intervals",
    transform=ax.transAxes, va="top",
    fontsize=10, color="#657083",
)

fig.suptitle(subtitle, fontsize=10, color="#778193")
save_figure(fig, "01_gap")

In [ ]:
# Figure 2: empirical gap distributions.

fig, ax = plt.subplots(figsize=(9, 5.2))

for index, steps in enumerate(plot_summary["n_steps"].astype(int)):
    gaps = np.sort(
        good.loc[good["n_steps"] == steps, "gap"].to_numpy()
    )
    fractions = np.arange(1, len(gaps) + 1) / len(gaps)

    ax.step(
        gaps, fractions, where="post",
        color=COLORS[index % len(COLORS)],
        linewidth=2.3,
        label=f"{steps} intervals · {len(gaps)} seeds",
    )

ax.set_xlabel("Rectangularity gap")
ax.set_ylabel("Fraction of seeds with gap ≤ x")
ax.set_ylim(0, 1.03)
ax.set_title("How broadly does the effect appear?", loc="left", pad=18)
ax.legend(frameon=False, loc="lower right")
ax.grid(axis="y", color="#ECEFF3")
ax.set_axisbelow(True)

fig.suptitle(subtitle, fontsize=10, color="#778193")
save_figure(fig, "02_distribution")

In [ ]:
# Figure 3: nonrectangular objective and runtime for every horizon (incl. 5-6 steps).
fig, ax = plt.subplots(figsize=(9, 5.2))
nr = summary.sort_values("n_steps")
ax.plot(nr["n_steps"], nr["mean_nonrectangular_value"], "o-", color=COLORS[0], label="nonrectangular")
if "mean_rectangular_value" in nr:
    ax.plot(nr["n_steps"], nr["mean_rectangular_value"], "o-", color=COLORS[3], label="rectangular")
ax.set_xticks(nr["n_steps"])
ax.set_xlabel("Trading intervals")
ax.set_ylabel("Mean objective")
ax.set_title(f"Objective by horizon (NEIGHBORS = {NEIGHBORS})", loc="left", pad=18)
ax.legend(frameon=False)
ax.grid(axis="y", color="#ECEFF3"); ax.set_axisbelow(True)
fig.suptitle(subtitle, fontsize=10, color="#778193")
save_figure(fig, "03_values")


In [ ]:
import shutil

# Make sure the latest tables are on disk.
results.to_csv(output_dir / "results.csv", index=False)
summary.to_csv(output_dir / "summary.csv", index=False)

archive_path = Path(
    shutil.make_archive(
        base_name=str(results_root / output_dir.name),
        format="zip",
        root_dir=str(output_dir.parent),
        base_dir=output_dir.name,
    )
)

print("Results folder:", output_dir.resolve())
print("ZIP archive:   ", archive_path.resolve())


## Final sweeps and figures
Results are cached in `final_results/` (CSV per sweep, per seed and seed-averaged), so re-running skips finished points. Figures show seed averages only.

In [ ]:
# ===== Final sweeps: budget, time steps, penalty (cached to final_results/) =====
FINAL_DIR = PROJECT_DIR / "final_results"
FINAL_DIR.mkdir(exist_ok=True)

FINAL_SEEDS = SEEDS[:20]
BUDGETS = [0, 0.25, 0.5, 1, 1.5, 2, 3]       # sweep at 3 intervals, penalty 10
PENALTIES = [1, 2, 5, 10, 20, 50]            # sweep at 3 intervals, budget 1
STEPS = [2, 3, 4]                            # budget 1, penalty 10
STEPS_NEIGHBORS = 4                          # sparse coupling for the 4-interval point
SWEEP_T = 3                                  # intervals used by budget/penalty sweeps


def _build(seed, n_steps, budget, penalty, neighbors):
    model = Heston(**MODEL_PARAMETERS)
    times = np.linspace(0.0, MATURITY, n_steps + 1)
    problem = SuperhedgingProblem(payoff=call_payoff, trading_times=times)
    disc = {"seed": int(seed), "reference_branches": BRANCHES}
    rect = FastRectangularSolver(model, times, budget, disc, neighbors=neighbors)
    outer = FastRelaxedSuperhedgingSolver(
        problem=problem,
        inner_solver=NonrectangularSolver(model, times, budget, disc),
        penalty=penalty,
    )
    return rect, outer


def _exact_pair(seed, n_steps, budget, penalty, neighbors):
    rect, outer = _build(seed, n_steps, budget, penalty, neighbors)
    nonrect_value = outer.solve_adaptive_hedge_nonrectangular(
        rect, CAPITAL_BOUNDS, HOLDING_BOUNDS)["value"]
    rect_value = outer.solve_adaptive_hedge_rectangular(
        rect, CAPITAL_BOUNDS, HOLDING_BOUNDS)["value"]
    return float(nonrect_value), float(rect_value)


def _first_order(seed, n_steps, budgets, penalty):
    """Budget-0 optimal hedge, then capital + penalty * (E[shortfall] + budget * sensitivity).
    Capital is re-optimised; holdings are held fixed when differentiating."""
    rect, outer = _build(seed, n_steps, 0.0, penalty, 1)
    res = outer.solve_adaptive_hedge_rectangular(rect, CAPITAL_BOUNDS, HOLDING_BOUNDS)
    adv, refs = rect.adversarial_paths, rect.reference_paths
    _, groups = outer._build_history_groups(adv)
    lookup = {np.ascontiguousarray(p).tobytes(): j for j, p in enumerate(adv)}
    idx = np.array([lookup[np.ascontiguousarray(p).tobytes()] for p in refs])
    H = np.stack([res["holdings"][res["offsets"][t] + groups[t][idx]]
                  for t in range(n_steps)], axis=1)

    S = refs[:, :, 0]
    payoff = np.maximum(S[:, -1] - STRIKE, 0.0)
    gains = (H * np.diff(S, axis=1)).sum(axis=1)
    w = rect.reference_probabilities

    d_gain = np.zeros_like(S)
    for k in range(1, n_steps):
        d_gain[:, k] = H[:, k - 1] - H[:, k]
    d_gain[:, n_steps] = H[:, n_steps - 1]
    d_payoff = np.zeros_like(S)
    d_payoff[:, n_steps] = S[:, n_steps] > STRIKE
    grad_sq = ((d_payoff - d_gain) ** 2).sum(axis=1)   # variance coordinate has zero gradient

    need = payoff - gains
    bp = np.clip(need, *CAPITAL_BOUNDS)
    xs = np.unique(np.concatenate([
        np.linspace(*CAPITAL_BOUNDS, 2001), bp, bp + 1e-9, list(CAPITAL_BOUNDS)]))
    shortfall = need[None, :] - xs[:, None]
    active = shortfall > 0
    mean_short = (np.maximum(shortfall, 0.0) * w).sum(axis=1)
    sens = np.sqrt((active * grad_sq * w).sum(axis=1))
    return {b: float((xs + penalty * (mean_short + b * sens)).min()) for b in budgets}


def _load(name):
    path = FINAL_DIR / f"{name}_by_seed.csv"
    return pd.read_csv(path) if path.exists() else pd.DataFrame(
        columns=["seed", "x", "nonrectangular", "rectangular", "first_order"])


def _save(name, df):
    df.to_csv(FINAL_DIR / f"{name}_by_seed.csv", index=False)
    (df.groupby("x")[["nonrectangular", "rectangular", "first_order"]]
       .mean().reset_index().to_csv(FINAL_DIR / f"{name}_mean.csv", index=False))


def _sweep(name, xs, compute):
    """compute(seed, x) -> (nonrect, rect, first_order). Resumes from the saved CSV."""
    df = _load(name)
    done = set(zip(df["seed"].astype(int), df["x"].astype(float)))
    rows = [df]
    for seed in FINAL_SEEDS:
        for x in xs:
            if (int(seed), float(x)) in done:
                continue
            n, r, f = compute(seed, x)
            rows.append(pd.DataFrame([dict(seed=seed, x=x, nonrectangular=n,
                                           rectangular=r, first_order=f)]))
        _save(name, pd.concat(rows, ignore_index=True))
        print(f"{name}: seed {seed} done", flush=True)
    return pd.concat(rows, ignore_index=True)


def _budget_point(seed, b):
    n, r = _exact_pair(seed, SWEEP_T, b, PENALTY, None)
    return n, r, _first_order(seed, SWEEP_T, [b], PENALTY)[b]


def _penalty_point(seed, k):
    n, r = _exact_pair(seed, SWEEP_T, BUDGET, float(k), None)
    return n, r, _first_order(seed, SWEEP_T, [BUDGET], float(k))[BUDGET]


def _steps_point(seed, T):
    n, r = _exact_pair(seed, T, BUDGET, PENALTY, STEPS_NEIGHBORS)
    return n, r, _first_order(seed, T, [BUDGET], PENALTY)[BUDGET]


budget_by_seed = _sweep("budget", BUDGETS, _budget_point)
steps_by_seed = _sweep("steps", STEPS, _steps_point)
penalty_by_seed = _sweep("penalty", PENALTIES, _penalty_point)
print("Saved to", FINAL_DIR.resolve())


In [ ]:
# ===== Final figures: seed averages only, saved to final_results/ =====
PURPLE, APRICOT, LILAC = "#5B2A86", "#F2A65A", "#B58DD6"
STYLE = {
    "nonrectangular": dict(color=PURPLE, ls="-"),
    "rectangular": dict(color=APRICOT, ls="-"),
    "first_order": dict(color=LILAC, ls="--"),
}
LABELS = {"nonrectangular": "nonrectangular", "rectangular": "rectangular",
          "first_order": "first-order"}
LABEL_COLOR = {"nonrectangular": PURPLE, "rectangular": "#D9822B", "first_order": LILAC}


def final_figure(name, xlabel, xticks=None, log_x=False):
    means = pd.read_csv(FINAL_DIR / f"{name}_mean.csv").sort_values("x")
    fig, ax = plt.subplots(figsize=(6.4, 4.4))
    for column in ["rectangular", "nonrectangular", "first_order"]:
        ax.plot(means["x"], means[column], lw=2.4, marker="o", ms=5, **STYLE[column])
        ax.annotate(LABELS[column], (means["x"].iloc[-1], means[column].iloc[-1]),
                    xytext=(8, 0), textcoords="offset points", va="center",
                    fontsize=11, color=LABEL_COLOR[column])
    ax.set_xlabel(xlabel)
    ax.set_ylabel("objective value")
    if log_x:
        ax.set_xscale("log")
        ax.set_xticks(means["x"])
        ax.set_xticklabels([f"{v:g}" for v in means["x"]])
        ax.minorticks_off()
    elif xticks is not None:
        ax.set_xticks(xticks)
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.yaxis.grid(True, color="#EEE8F3")
    ax.set_axisbelow(True)
    ax.margins(x=0.04)
    fig.subplots_adjust(right=0.78, bottom=0.15)
    for ext in ("png", "pdf"):
        fig.savefig(FINAL_DIR / f"{name}.{ext}", dpi=300, bbox_inches="tight")
    plt.show()


final_figure("budget", "budget ε")
final_figure("steps", "trading intervals", xticks=STEPS)
final_figure("penalty", "penalty κ", log_x=True)
